In [ ]:
# final_hybrid_pipeline_fd002_004.py
"""
Hybrid model (Option B) final pipeline with fixes:
- trains on FD002, FD003, FD004 training sets
- CNN -> BiLSTM -> Transformer
- MC-Dropout for Bayesian approx (stochastic forward passes)
- SHAP DeepExplainer (fallback to KernelExplainer)
- fixes: robust loading, keep engine ids, consistent scaling + unscaling
"""

import os
import random
import numpy as np
import pandas as pd
from glob import glob
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

import shap

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

WINDOW = 50                 # input time window
POOL_SIZE = 2               # MaxPooling factor (reduces time to WINDOW/POOL_SIZE)
BATCH_SIZE = 64
EPOCHS = 80
MC_SAMPLES = 50             # T for MC-dropout
SHAP_BG = 100               # DeepExplainer background size
SHAP_EXPLAIN = 50           # number of validation samples to explain
TRAIN_FILES = ['train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt']
OUT_DIR = 'artifacts'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'plots'), exist_ok=True)

#  helper: robust loader
def load_fd_train(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    # read robustly using regex separator (handles variable spaces)
    df = pd.read_csv(path, sep=r'\s+', header=None)
    if df.shape[1] < 26:
        raise ValueError(f"Expected >=26 columns in {path}, got {df.shape[1]}")
    df = df.iloc[:, :26].copy()
    columns = ['engine_id', 'cycle', 'setting1', 'setting2', 'setting3'] + ['s'+str(i) for i in range(1,22)]
    df.columns = columns
    #  RUL
    max_cycles = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id','max_cycle']
    df = df.merge(max_cycles, on='engine_id', how='left')
    df['RUL'] = df['max_cycle'] - df['cycle']
    df.drop(columns=['max_cycle'], inplace=True)
    return df

#  load and concat
dfs = [load_fd_train(f) for f in TRAIN_FILES]
full_df = pd.concat(dfs, ignore_index=True)
print("Combined train shape:", full_df.shape)

#  preprocess
sensor_cols = ['s'+str(i) for i in range(1,22)]
setting_cols = ['setting1','setting2','setting3']
cols_to_normalize = setting_cols + sensor_cols

scaler = MinMaxScaler()
full_df[cols_to_normalize] = scaler.fit_transform(full_df[cols_to_normalize])

#  windowing (keep engine ids)
def create_windows_with_ids(df, window_size=WINDOW, cols=cols_to_normalize):
    X, y, eids, cycles = [], [], [], []
    for eid in df['engine_id'].unique():
        ed = df[df['engine_id'] == eid].reset_index(drop=True)
        feats = ed[cols].values
        rul = ed['RUL'].values
        cyc = ed['cycle'].values
        if feats.shape[0] < window_size:
            continue
        for i in range(feats.shape[0] - window_size + 1):
            X.append(feats[i:i+window_size])
            y.append(rul[i+window_size-1])
            eids.append(eid)
            cycles.append(cyc[i+window_size-1])
    return np.array(X), np.array(y), np.array(eids), np.array(cycles)

X, y, eids, cycles = create_windows_with_ids(full_df, WINDOW, cols_to_normalize)
print("X shape:", X.shape, "y shape:", y.shape)

# store raw y (unscaled) and compute scaled version
y_unscaled = y.copy()
# optional clipping: y_unscaled = np.clip(y_unscaled, 0, 130)
max_rul = y_unscaled.max()
y_scaled = y_unscaled / (max_rul + 1e-9)

#  create reduced X for pooling (we will MaxPool in model; easier to downsample here for clarity)
# To match the model which pools by factor POOL_SIZE, we can downsample by taking every POOL_SIZE-th step.
# This keeps the same semantics as MaxPooling1D(pool_size=2) for many tasks.
X_reduced = X[:, ::POOL_SIZE, :]  # shape (N, WINDOW/POOL_SIZE, F)
print("X_reduced shape (after downsample):", X_reduced.shape)

#  train-val split (preserve engine alignment)
X_tr, X_val, y_tr, y_val, eids_tr, eids_val = train_test_split(
    X_reduced, y_scaled, eids, test_size=0.15, random_state=SEED, shuffle=True
)

# Save eids for later plotting
np.save(os.path.join(OUT_DIR, 'eids_val.npy'), eids_val)

#  model building
def positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dims//2)) / np.float32(d_model))
    angle_rads = positions * angle_rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
    pe[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(pe[np.newaxis, ...], tf.float32)  # shape (1, seq_len, d_model)

class PositionalEncodingLayer(layers.Layer):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pe = positional_encoding(seq_len, d_model)
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

def transformer_encoder_block(x, head_size=16, num_heads=2, ff_dim=128, dropout=0.1, d_model=None):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size, dropout=dropout)(x, x)
    attn = layers.Dropout(dropout)(attn)
    out1 = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    ff = layers.Conv1D(filters=ff_dim, kernel_size=1, activation='relu')(out1)
    ff = layers.Dropout(dropout)(ff)
    if d_model is None:
        raise ValueError("d_model must be provided as int")
    ff = layers.Conv1D(filters=d_model, kernel_size=1)(ff)
    out2 = layers.LayerNormalization(epsilon=1e-6)(out1 + ff)
    return out2

def build_hybrid_model(input_shape, dropout_rate=0.2):
    # input_shape: (seq_len_reduced, n_features)
    seq_len, n_feat = input_shape
    inp = Input(shape=(seq_len, n_feat), name='input_layer')
    # CNN block
    x = layers.Conv1D(filters=64, kernel_size=3, padding='same', activation='relu')(inp)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Conv1D(filters=64, kernel_size=3, padding='same', activation='relu')(x)
    # pooling already applied via downsampling; keep this optional: commented out
    # x = layers.MaxPooling1D(pool_size=2)(x)

    # BiLSTM block
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(dropout_rate)(x)

    # project to d_model
    d_model = n_feat
    x = layers.Conv1D(filters=d_model, kernel_size=1, activation='linear')(x)

    # positional encoding
    x = PositionalEncodingLayer(seq_len, d_model)(x)

    # transformer blocks
    for _ in range(2):
        x = transformer_encoder_block(x, head_size=16, num_heads=2, ff_dim=128, dropout=dropout_rate, d_model=d_model)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    out = layers.Dense(1, activation='linear', name='rul')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mse',
                  metrics=[tf.keras.metrics.MeanAbsoluteError(), tf.keras.metrics.RootMeanSquaredError()])
    return model

input_shape = (X_reduced.shape[1], X_reduced.shape[2])
model = build_hybrid_model(input_shape, dropout_rate=0.2)
model.summary()

#  callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUT_DIR, 'best_hybrid_fd002_004.keras'), save_best_only=True, monitor='val_loss')
]

#  training
history = model.fit(X_tr, y_tr,
                    validation_data=(X_val, y_val),
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    callbacks=callbacks,
                    verbose=2)

# Save scaler and some arrays for reproducibility
joblib.dump(scaler, os.path.join(OUT_DIR, 'scaler_fd002_004.save'))
np.save(os.path.join(OUT_DIR, 'X_val.npy'), X_val)
np.save(os.path.join(OUT_DIR, 'y_val_scaled.npy'), y_val)
np.save(os.path.join(OUT_DIR, 'y_unscaled.npy'), y_unscaled)  # full array
np.save(os.path.join(OUT_DIR, 'eids.npy'), eids)  # full eids

#  MC-Dropout predictions (stochastic)
def mc_predict_unscaled(model, X, max_rul, T=MC_SAMPLES, batch_size=256):
    samples = []
    for t in range(T):
        # attempt stochastic forward pass using training=True (this keeps dropout active)
        try:
            outs = model(X, training=True).numpy().ravel()
        except Exception:
            # fallback to predict (may be deterministic), still include
            outs = model.predict(X, batch_size=batch_size, verbose=0).ravel()
        samples.append(outs)
    samples = np.stack(samples, axis=0)  # T x N (scaled targets)
    mean_scaled = samples.mean(axis=0)
    std_scaled = samples.std(axis=0)
    return mean_scaled * max_rul, std_scaled * max_rul, samples  # return unscaled mean, std, and scaled samples

print("Running MC-Dropout on validation set ...")
mean_val_unscaled, std_val_unscaled, samples_scaled = mc_predict_unscaled(model, X_val, max_rul, T=MC_SAMPLES, batch_size=256)

# get y_val_unscaled aligned with validation indices
# y_val we used during training was scaled. Unscale it for metric computation:
y_val_unscaled = y_val * max_rul

mae_val = mean_absolute_error(y_val_unscaled, mean_val_unscaled)
rmse_val = np.sqrt(np.mean((y_val_unscaled - mean_val_unscaled) ** 2))
print(f"Validation MAE (cycles): {mae_val:.3f}, RMSE (cycles): {rmse_val:.3f}")

# Save MC outputs
np.save(os.path.join(OUT_DIR, 'mc_val_mean_unscaled.npy'), mean_val_unscaled)
np.save(os.path.join(OUT_DIR, 'mc_val_std_unscaled.npy'), std_val_unscaled)
np.save(os.path.join(OUT_DIR, 'mc_samples_scaled.npy'), samples_scaled)

#  per-engine plots: choose top-5 worst engines by MAE on val
eids_val = np.load(os.path.join(OUT_DIR, 'eids_val.npy'))
from collections import defaultdict
engine_indices = defaultdict(list)
for idx, eid in enumerate(eids_val):
    engine_indices[eid].append(idx)

engine_mae = {}
for eid, idxs in engine_indices.items():
    engine_mae[eid] = np.mean(np.abs(y_val_unscaled[idxs] - mean_val_unscaled[idxs]))

top5 = sorted(engine_mae.items(), key=lambda x: x[1], reverse=True)[:5]
top5_eids = [t[0] for t in top5]
print("Top 5 worst validation engines:", top5_eids)

for eid in top5_eids:
    idxs = engine_indices[eid]
    times = np.arange(len(idxs))
    y_true = y_val_unscaled[idxs]
    y_pred = mean_val_unscaled[idxs]
    y_std = std_val_unscaled[idxs]
    plt.figure(figsize=(8,4))
    plt.plot(times, y_true, label='True RUL', marker='o')
    plt.plot(times, y_pred, label='Predicted mean', marker='o')
    plt.fill_between(times, y_pred - 2*y_std, y_pred + 2*y_std, alpha=0.25, label='±2σ')
    plt.title(f'Engine {eid} — Predicted vs True RUL (MAE={engine_mae[eid]:.1f} cycles)')
    plt.xlabel('Window index (time-ordered)')
    plt.ylabel('RUL (cycles)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'plots', f'engine_{eid}_rul_pred.png'))
    plt.close()

#  SHAP interpretability (DeepExplainer preferred)
print("Running SHAP DeepExplainer (background size=%d) ..." % SHAP_BG)
# pick background from training set (X_tr), and subset to manageable size
bg_idx = np.random.choice(np.arange(X_tr.shape[0]), size=min(SHAP_BG, X_tr.shape[0]), replace=False)
background = X_tr[bg_idx]

# pick explain samples from validation set
explain_idx = np.random.choice(np.arange(X_val.shape[0]), size=min(SHAP_EXPLAIN, X_val.shape[0]), replace=False)
X_explain = X_val[explain_idx]

# DeepExplainer expects TF/Keras model - ensure our model returns single scalar
try:
    explainer = shap.DeepExplainer(model, background)
    shap_vals = explainer.shap_values(X_explain)
    # shap_vals can be list (for outputs) or array
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[0]
    shap_arr = np.array(shap_vals)  # shape (N_explain, seq, feat)
except Exception as e:
    print("DeepExplainer failed (falling back to KernelExplainer). Error:", e)
    # fallback: flatten and use KernelExplainer with small background
    # Flatten function
    def predict_unscaled_flat(X_flat):
        X3 = X_flat.reshape((X_flat.shape[0], X_val.shape[1], X_val.shape[2]))
        preds = model.predict(X3, batch_size=128).ravel()
        return preds * max_rul
    bg_flat = background.reshape((background.shape[0], -1))
    ex_flat = X_explain.reshape((X_explain.shape[0], -1))
    ke = shap.KernelExplainer(predict_unscaled_flat, bg_flat[:min(50, bg_flat.shape[0])])
    shap_vals_flat = ke.shap_values(ex_flat, nsamples=200)
    shap_arr = np.array(shap_vals_flat).reshape((len(shap_vals_flat), X_val.shape[1], X_val.shape[2]))

# Aggregate SHAP over time to per-sensor importance
mean_abs_shap = np.mean(np.abs(shap_arr), axis=0)  # (seq, feat)
sensor_importance = mean_abs_shap.sum(axis=0)      # (feat,)
sensor_importance = sensor_importance / np.sum(sensor_importance)

imp_df = pd.DataFrame({'feature': cols_to_normalize, 'importance': sensor_importance})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)
imp_df.to_csv(os.path.join(OUT_DIR, 'sensor_shap_importance_deepexp.csv'), index=False)

# plot sensor importance
plt.figure(figsize=(8,6))
plt.barh(imp_df['feature'], imp_df['importance'])
plt.gca().invert_yaxis()
plt.title('Sensor importance (SHAP aggregated over time)')
plt.xlabel('Normalized importance')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'plots', 'sensor_shap_importance_deepexp.png'))
plt.close()

print("Top sensors by SHAP (aggregated):")
print(imp_df.head(10))

#  save final model and artifacts
model.save(os.path.join(OUT_DIR, 'hybrid_cnn_bilstm_transformer_fd002_004.keras'))
print("Saved model and artifacts to", OUT_DIR)

#  final summary prints
print("\nSummary:")
print(f"- Combined train windows: {X.shape[0]}")
print(f"- Validation windows: {X_val.shape[0]}")
print(f"- max_rul used for scaling: {max_rul}")
print(f"- Validation MAE (cycles): {mae_val:.3f}")
print(f"- Validation RMSE (cycles): {rmse_val:.3f}")
print("- Artifacts saved under:", OUT_DIR)
print("- Per-engine plots saved in:", os.path.join(OUT_DIR, 'plots'))


Combined train shape: (139728, 27)
X shape: (126988, 50, 24) y shape: (126988,)
X_reduced shape (after downsample): (126988, 25, 24)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 25, 24)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 25, 64)    │      4,672 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 25, 64)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 25, 64)    │     12,352 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 25, 128)   │     66,048 │ conv1d_1[0][0]    │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 25, 128)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 25, 24)    │      3,096 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 25, 24)    │          0 │ conv1d_2[0][0]    │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 25, 24)    │      3,192 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 25, 24)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 25, 24)    │          0 │ positional_encod… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 25, 24)    │         48 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 25, 128)   │      3,200 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 25, 128)   │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 25, 24)    │      3,096 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 25, 24)    │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 25, 24)    │         48 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 25, 24)    │      3,192 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 25, 24)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 25, 24)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_6[0][0] 

 Total params: 108,665 (424.47 KB)

 Trainable params: 108,665 (424.47 KB)

...

Epoch 1/80
1687/1687 - 47s - 28ms/step - loss: 0.0279 - mean_absolute_error: 0.1312 - root_mean_squared_error: 0.1671 - val_loss: 0.0243 - val_mean_absolute_error: 0.1256 - val_root_mean_squared_error: 0.1560 - learning_rate: 1.0000e-04
Epoch 2/80
1687/1687 - 33s - 20ms/step - loss: 0.0243 - mean_absolute_error: 0.1219 - root_mean_squared_error: 0.1560 - val_loss: 0.0190 - val_mean_absolute_error: 0.1061 - val_root_mean_squared_error: 0.1378 - learning_rate: 1.0000e-04
Epoch 3/80
1687/1687 - 41s - 24ms/step - loss: 0.0203 - mean_absolute_error: 0.1087 - root_mean_squared_error: 0.1426 - val_loss: 0.0148 - val_mean_absolute_error: 0.0873 - val_root_mean_squared_error: 0.1219 - learning_rate: 1.0000e-04
Epoch 4/80
1687/1687 - 33s - 20ms/step - loss: 0.0165 - mean_absolute_error: 0.0959 - root_mean_squared_error: 0.1286 - val_loss: 0.0122 - val_mean_absolute_error: 0.0810 - val_root_mean_squared_error: 0.1103 - learning_rate: 1.0000e-04
Epoch 5/80
1687/1687 - 41s - 24ms/step - loss: 0.014

/usr/local/lib/python3.12/dist-packages/shap/explainers/_deep/deep_tf.py:94: UserWarning: Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(100, 25, 24))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(200, 25, 24))']
  warnings.warn(msg)


DeepExplainer failed (falling back to KernelExplainer). Error: in user code:

    File "/usr/local/lib/python3.12/dist-packages/shap/explainers/_deep/deep_tf.py", line 265, in grad_graph  *
        x_grad = tape.gradient(out, shap_rAnD)

    LookupError: gradient registry has no entry for: shap_Neg

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 736ms/step


  0%|          | 0/50 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 697ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/ste

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


85/85 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
test_FD002.txt MAE (cycles) on official test: 20.747
test_FD002.txt RMSE (cycles) on official test: 29.392


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
test_FD003.txt MAE (cycles) on official test: 24.943
test_FD003.txt RMSE (cycles) on official test: 36.234


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
test_FD004.txt MAE (cycles) on official test: 28.115
test_FD004.txt RMSE (cycles) on official test: 39.449
